# YOLO11 LEGO Brick Detector — Kaggle Training

Trains a single-class LEGO brick detector using YOLO11m on the Hex:Lego dataset (8,320 images).

**Setup:** Enable GPU in Kaggle: Settings > Accelerator > GPU T4 x2

**Output:** `lego_yolo11.pt` — download from the Output tab when complete.

In [ ]:
# Install dependencies
!pip install -q ultralytics roboflow

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
        print(f"  GPU {i}: {props.name} ({mem / 1024**3:.1f} GB)")
else:
    print("WARNING: No GPU detected! Enable GPU in Settings > Accelerator")

In [ ]:
# Download the Hex:Lego dataset (8,320 images, 28 classes)
from roboflow import Roboflow

rf = Roboflow(api_key="qIs5Ph2rRMPC1SMMj9Bc")
project = rf.workspace("hexhewwie").project("hex-lego")
version = project.version(3)
dataset = version.download("yolov8", location="/kaggle/working/hex-lego")

print(f"\nDataset downloaded to: {dataset.location}")

In [ ]:
# Remap all 28 classes to single class 0 ("lego")
# YOLO's job is just to find bricks — Brickognize API handles identification

import os
from pathlib import Path

dataset_dir = Path("/kaggle/working/hex-lego")
remapped = 0

for split in ["train", "valid", "test"]:
    labels_dir = dataset_dir / split / "labels"
    if not labels_dir.exists():
        continue
    for label_file in labels_dir.glob("*.txt"):
        lines = label_file.read_text().strip().split("\n")
        new_lines = []
        for line in lines:
            if not line.strip():
                continue
            parts = line.strip().split()
            # Replace class ID with 0
            parts[0] = "0"
            new_lines.append(" ".join(parts))
        label_file.write_text("\n".join(new_lines) + "\n")
        remapped += 1

print(f"Remapped {remapped} label files to single class")

In [ ]:
# Write updated data.yaml with single class
import yaml

dataset_dir = Path("/kaggle/working/hex-lego")

data_yaml = {
    "train": str(dataset_dir / "train" / "images"),
    "val": str(dataset_dir / "valid" / "images"),
    "test": str(dataset_dir / "test" / "images"),
    "nc": 1,
    "names": ["lego"],
}

yaml_path = dataset_dir / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"Wrote {yaml_path}")
print(yaml.dump(data_yaml, default_flow_style=False))

In [ ]:
# Quick sanity check — count images and labels
for split in ["train", "valid", "test"]:
    imgs = list((dataset_dir / split / "images").glob("*"))
    lbls = list((dataset_dir / split / "labels").glob("*.txt"))
    print(f"{split}: {len(imgs)} images, {len(lbls)} labels")

In [ ]:
# Train YOLO11m (first run)
from ultralytics import YOLO

model = YOLO("yolo11m.pt")  # pretrained on COCO

results = model.train(
    data=str(dataset_dir / "data.yaml"),
    epochs=60,
    patience=12,           # early stopping if no improvement for 12 epochs
    imgsz=640,
    batch=32,              # 16 per GPU (auto-batch not supported with multi-GPU)
    device=[0, 1],         # use both T4 GPUs
    workers=4,
    project="/kaggle/working/runs",
    name="lego_yolo11",
    save_period=10,        # save checkpoint every 10 epochs (epoch10.pt, epoch20.pt, etc.)
    
    # Augmentation — robust to real-world photos
    augment=True,
    hsv_h=0.015,           # hue variation (different brick colors)
    hsv_s=0.5,             # saturation variation (lighting)
    hsv_v=0.4,             # brightness variation
    degrees=15,            # rotation (bricks at angles)
    translate=0.1,
    scale=0.5,             # scale variation (bricks at different distances)
    flipud=0.0,            # no vertical flip (bricks don't fly)
    fliplr=0.5,            # horizontal flip
    mosaic=1.0,            # mosaic augmentation (multiple bricks per scene)
    mixup=0.1,             # slight mixup for robustness
    copy_paste=0.2,        # copy-paste augmentation (handles occlusion)
    erasing=0.2,           # random erasing (handles partial visibility)
    close_mosaic=8,        # disable mosaic for last 8 epochs (fine-tune)
    
    # Optimizer
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,              # final LR = lr0 * lrf
    weight_decay=0.0005,
    warmup_epochs=3,
    
    verbose=True,
    plots=True,
)

In [ ]:
# Save checkpoints to /kaggle/working/ output so they persist between sessions
# Run this cell BEFORE your session expires, or periodically during training.
#
# After your session ends:
#   1. Go to the notebook's Output tab and download last.pt (or best.pt)
#   2. Create a new Kaggle Dataset and upload the .pt file
#   3. In your next session, add that dataset via "Add Data" on the right panel
#   4. Update the checkpoint path in the Resume cell below to point to your dataset
#      e.g. /kaggle/input/your-dataset-name/last.pt

import shutil
from pathlib import Path

weights_dir = Path("/kaggle/working/runs/lego_yolo11/weights")
output_dir = Path("/kaggle/working")

copied = []
for name in ["last.pt", "best.pt"]:
    src = weights_dir / name
    if src.exists():
        dst = output_dir / f"checkpoint_{name}"
        shutil.copy2(src, dst)
        size_mb = dst.stat().st_size / (1024 * 1024)
        copied.append(f"  {dst} ({size_mb:.1f} MB)")

# Also copy any epoch snapshots (epoch10.pt, epoch20.pt, etc.)
for src in sorted(weights_dir.glob("epoch*.pt")):
    dst = output_dir / f"checkpoint_{src.name}"
    shutil.copy2(src, dst)
    size_mb = dst.stat().st_size / (1024 * 1024)
    copied.append(f"  {dst} ({size_mb:.1f} MB)")

if copied:
    print("Checkpoints saved to output directory:")
    print("\n".join(copied))
    print("\nThese will appear in the Output tab after the session ends.")
else:
    print("No checkpoints found yet — training hasn't started or no weights saved.")

In [ ]:
# RESUME training after Kaggle session restart
# Only run this cell if your previous session was killed mid-training.
# Skip this cell on first run.
#
# If you uploaded your checkpoint as a Kaggle dataset, update this path:
UPLOADED_CHECKPOINT = None  # e.g. "/kaggle/input/your-dataset-name/checkpoint_last.pt"

from pathlib import Path
from ultralytics import YOLO

# Check for checkpoint: uploaded dataset first, then local
candidates = [
    UPLOADED_CHECKPOINT,
    "/kaggle/working/runs/lego_yolo11/weights/last.pt",
    "/kaggle/working/checkpoint_last.pt",
]

checkpoint = None
for path in candidates:
    if path and Path(path).exists():
        checkpoint = path
        break

if checkpoint:
    print(f"Found checkpoint: {checkpoint}")
    model = YOLO(checkpoint)
    results = model.train(resume=True)
    print("Training resumed and completed!")
else:
    print("No checkpoint found — run the training cell above instead.")
    print("If you uploaded a checkpoint dataset, set UPLOADED_CHECKPOINT above.")

In [ ]:
# Evaluate on test set
best_model = YOLO("/kaggle/working/runs/lego_yolo11/weights/best.pt")

metrics = best_model.val(
    data=str(dataset_dir / "data.yaml"),
    split="test",
)

print(f"\n{'='*50}")
print(f"Test Results:")
print(f"  mAP@0.5:      {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"  Precision:     {metrics.box.mp:.4f}")
print(f"  Recall:        {metrics.box.mr:.4f}")
print(f"{'='*50}")

In [ ]:
# Copy best weights to output directory for easy download
import shutil

src = Path("/kaggle/working/runs/lego_yolo11/weights/best.pt")
dst = Path("/kaggle/working/lego_yolo11.pt")
shutil.copy2(src, dst)

size_mb = dst.stat().st_size / (1024 * 1024)
print(f"\nModel saved: {dst}")
print(f"Size: {size_mb:.1f} MB")
print(f"\nDownload from the Output tab on the right panel.")
print(f"Then place it in your lego_classifier/ folder as 'lego_yolo11.pt'")

In [ ]:
# Also save the training plots
from IPython.display import Image as IPImage, display

plots_dir = Path("/kaggle/working/runs/lego_yolo11")

for plot_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.png"]:
    plot_path = plots_dir / plot_name
    if plot_path.exists():
        print(f"\n{plot_name}:")
        display(IPImage(filename=str(plot_path), width=800))